<a href="https://colab.research.google.com/github/Capitagno/MeiboSuite/blob/main/meibomio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Utility: estrazione impronta ottica target
import cv2
import numpy as np
import os
from google.colab import drive

# Monta il drive se non è già attivo
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

def calcola_impronta_ottica(nome_file):
    """
    Calcola media e deviazione standard per l'histogram matching.
    """
    # Percorso aggiornato alla tua nuova cartella
    percorso_base = '/content/drive/MyDrive/Meibo_Analisi'
    percorso_completo = os.path.join(percorso_base, nome_file)

    img = cv2.imread(percorso_completo, cv2.IMREAD_GRAYSCALE)

    if img is None:
        print(f"Errore: impossibile caricare l'immagine da {percorso_completo}")
        return None, None

    media = np.mean(img)
    std_dev = np.std(img)

    print(f"--- Parametri target estratti ---")
    print(f"Media (luminosità): {media:.2f}")
    print(f"Deviazione standard (contrasto): {std_dev:.2f}")

    return media, std_dev

# Inserisci qui il nome esatto della tua foto ideale ritagliata
nome_foto_ideale = 'immagine_riferimento.jpeg'
media_target, std_target = calcola_impronta_ottica(nome_foto_ideale)

In [ ]:
# @title Utility 2 - Diagnostica fotometrica
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import drive

if not os.path.exists('/content/drive'): drive.mount('/content/drive')

# Inserisci il nome della foto che fallisce
nome_foto_problematica = '8.jpg'
percorso = os.path.join('/content/drive/MyDrive/MeibomioGrafia/immagini_IR', nome_foto_problematica)

img_raw = cv2.imread(percorso, cv2.IMREAD_GRAYSCALE)

if img_raw is None:
    print("Errore: impossibile caricare l'immagine.")
else:
    # 1. Estrazione dati originali
    media_orig = np.mean(img_raw)
    std_orig = np.std(img_raw)
    print(f"--- DIAGNOSTICA FOTOMETRICA ---")
    print(f"Media Originale: {media_orig:.2f}")
    print(f"Dev. Standard Originale: {std_orig:.2f}")

    # 2. Simulazione Equalizzazione Target (quella del nostro algoritmo)
    mu_target = 162.84
    sigma_target = 44.95
    sigma_sicuro = std_orig if std_orig > 0 else 1.0

    img_eq = (img_raw - media_orig) * (sigma_target / sigma_sicuro) + mu_target
    img_eq = np.clip(img_eq, 0, 255).astype(np.uint8)

    # 3. Simulazione Diga Grezza
    kernel_diga = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (45, 45))
    diga_chiusa = cv2.morphologyEx(img_eq, cv2.MORPH_CLOSE, kernel_diga)
    _, m_red_base = cv2.threshold(diga_chiusa, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 4. Rappresentazione Visiva del fallimento
    fig, axs = plt.subplots(1, 4, figsize=(22, 5))

    axs[0].imshow(img_raw, cmap='gray')
    axs[0].set_title('1. Immagine Originale')

    axs[1].hist(img_raw.ravel(), 256, [0,256], color='blue')
    axs[1].set_title('2. Istogramma Originale')

    axs[2].imshow(img_eq, cmap='gray')
    axs[2].set_title('3. Effetto Equalizzazione Target')

    axs[3].imshow(m_red_base, cmap='gray')
    axs[3].set_title('4. Diga Grezza (Risultato Otsu)')

    for ax in axs:
        if ax != axs[1]: ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# @title Utility - Calcolo Mappa della luce
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import drive

if not os.path.exists('/content/drive'): drive.mount('/content/drive')

# Inserisci il nome della foto con la luce sbilanciata
nome_foto_problematica = '8.jpg'
percorso = os.path.join('/content/drive/MyDrive/MeibomioGrafia/immagini_IR', nome_foto_problematica)

img_raw = cv2.imread(percorso, cv2.IMREAD_GRAYSCALE)

if img_raw is not None:
    # 1. Calcolo della Mappa della Luce
    img_float = img_raw.astype(np.float32)
    mappa_luce = cv2.GaussianBlur(img_float, (301, 301), 0)

    # 2. Correzione Flat-Field per SOTTRAZIONE (mantiene la gamma dinamica visibile)
    media_originale = np.mean(img_float)
    img_piatta = img_float - mappa_luce + media_originale
    img_piatta = np.clip(img_piatta, 0, 255).astype(np.uint8)

    # 3. Estrazione dettagli locali col CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_finale = clahe.apply(img_piatta)

    # 4. Controllo Diga
    _, otsu_finale = cv2.threshold(img_finale, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Rappresentazione Visiva
    fig, axs = plt.subplots(1, 4, figsize=(22, 5))

    axs[0].imshow(img_raw, cmap='gray')
    axs[0].set_title('1. Originale (Sbilanciata)')

    axs[1].imshow(mappa_luce, cmap='gray')
    axs[1].set_title('2. Mappa della Luce')

    axs[2].imshow(img_finale, cmap='gray')
    axs[2].set_title('3. Luce Sottratta + CLAHE')

    axs[3].imshow(otsu_finale, cmap='gray')
    axs[3].set_title('4. Nuova Diga (Otsu)')

    for ax in axs: ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Errore: impossibile caricare l'immagine.")

In [ ]:
# @title Diagnostica diga
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import drive

if not os.path.exists('/content/drive'): drive.mount('/content/drive')

nome_foto = '3.jpg'
percorso = os.path.join('/content/drive/MyDrive/MeibomioGrafia/immagini_IR', nome_foto)

img_raw = cv2.imread(percorso, cv2.IMREAD_GRAYSCALE)

if img_raw is not None:
    # 1. correzione flat-field
    img_float = img_raw.astype(np.float32)
    mappa_luce = cv2.GaussianBlur(img_float, (301, 301), 0)
    img_piatta = img_float - mappa_luce + np.mean(img_float)

    # 2. compressione forte del contrasto (parametro sigma abbassato a 25.0)
    mu_target = 180.00
    sigma_target = 40.0

    mu_old = np.mean(img_piatta)
    sigma_old = np.std(img_piatta)
    if sigma_old == 0: sigma_old = 1.0

    img_eq = (img_piatta - mu_old) * (sigma_target / sigma_old) + mu_target
    img_eq = np.clip(img_eq, 0, 255).astype(np.uint8)

    # 3. coperta morfologica gigante (parametro kernel alzato a 91)
    dim_kernel = 70
    kernel_diga = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dim_kernel, dim_kernel))
    diga_chiusa = cv2.morphologyEx(img_eq, cv2.MORPH_CLOSE, kernel_diga)

    _, m_red_base = cv2.threshold(diga_chiusa, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # tracciamento diagnostico
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))

    axs[0].imshow(img_raw, cmap='gray')
    axs[0].set_title('originale')
    axs[0].axis('off')

    axs[1].imshow(img_eq, cmap='gray')
    axs[1].set_title(f'equalizzata (contrasto {sigma_target})')
    axs[1].axis('off')

    axs[2].imshow(m_red_base, cmap='gray')
    axs[2].set_title(f'diga con kernel {dim_kernel}x{dim_kernel}')
    axs[2].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("errore: file non trovato.")

In [ ]:
# @title Prima versione Deep Learning
# 1. INSTALLAZIONE LIBRERIE DEEP LEARNING
# segmentation-models-pytorch è lo standard per usare U-Net professionali
!pip install -q segmentation-models-pytorch

import torch
import segmentation_models_pytorch as smp
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display, clear_output

# 2. COLLEGAMENTO GOOGLE DRIVE
print("🔄 Collegamento a Google Drive in corso...")
drive.mount('/content/drive')
print("✅ Drive collegato! Ora potremo scegliere i file.")

# Crea una cartella di lavoro se non esiste
work_dir = '/content/drive/MyDrive/Meibo_Analisi'
os.makedirs(work_dir, exist_ok=True)
print(f"📂 Cartella di lavoro: {work_dir} (Metti qui le tue immagini)")

In [ ]:
# @title v3.1-alpha
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive, output
import ipywidgets as widgets
import base64
from IPython.display import display, clear_output

# 1. Setup e caricamento
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
work_dir = '/content/drive/MyDrive/MeibomioGrafia/immagini_IR'
file_list = sorted([f for f in os.listdir(work_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

for i, f in enumerate(file_list): print(f"[{i}] {f}")
scelta = int(input("\n👉 Simone, inserisci il numero della foto: "))
path_img = os.path.join(work_dir, file_list[scelta])

img_raw = cv2.imread(path_img)
h_orig, w_orig = img_raw.shape[:2]
w_an = 1024
ratio_an = w_an / w_orig
img_an = cv2.resize(img_raw, (w_an, int(h_orig * ratio_an)))
gray_an = cv2.cvtColor(img_an, cv2.COLOR_BGR2GRAY)

# Pre-processing leggero per i muri
_, gray_ready = cv2.threshold(gray_an, 235, 235, cv2.THRESH_TRUNC)
gray_ready = cv2.medianBlur(gray_ready, 5)

# Display a 750px
w_disp = 750
ratio_disp = w_disp / w_an
img_disp_rgb = cv2.cvtColor(img_an, cv2.COLOR_BGR2RGB)
img_disp = cv2.resize(img_disp_rgb, (w_disp, int(img_disp_rgb.shape[0] * ratio_disp)))

# Muri automatici con fallback di sicurezza
try:
    blur_m = cv2.GaussianBlur(gray_ready, (21, 21), 0)
    lx = np.mean(blur_m, axis=0); xm = np.argmax(lx); sx_v = np.max(lx) * 0.70
    l_a, r_a = 0, w_an-1
    for x in range(xm, 0, -1):
        if lx[x] < sx_v: l_a = x; break
    for x in range(xm, w_an):
        if lx[x] < sx_v: r_a = x; break

    fy = np.mean(blur_m[:, l_a:r_a], axis=1); py = np.gradient(fy.astype(float))
    yp = np.argmax(fy); msop, msot = np.argmax(py[:yp]), np.argmin(py[yp:]) + yp
    alt = msot - msop
    m_t_an, m_b_an = int(msop + int(alt*0.07)), int(msot - int(alt*0.05))
except Exception:
    print("Avviso: immagine complessa, applico muri standard.")
    m_t_an, m_b_an = int(gray_ready.shape[0]*0.2), int(gray_ready.shape[0]*0.8)
    l_a, r_a = int(w_an*0.2), int(w_an*0.8)

m_t, m_b = int(m_t_an * ratio_disp), int(m_b_an * ratio_disp)
m_l, m_r = int(l_a * ratio_disp), int(r_a * ratio_disp)

app_out = widgets.Output()

def mostra_schermata_muri():
    _, buffer = cv2.imencode('.jpg', cv2.cvtColor(img_disp, cv2.COLOR_RGB2BGR))
    img_64 = base64.b64encode(buffer).decode('utf-8')
    html = f"""
    <div style="border: 4px solid #444; padding: 10px; display: inline-block; background: #1a1a1a; border-radius: 10px;">
        <canvas id="meibo"></canvas>
        <div style="margin-top: 10px;">
            <button id="run" style="padding: 15px 30px; background: #007bff; color: white; border: none; cursor: pointer; border-radius: 5px; width: 100%;">
                Avvia analisi morfologica
            </button>
        </div>
    </div>
    <script>
        var c = document.getElementById('meibo'); var ctx = c.getContext('2d');
        var img = new Image(); img.src = "data:image/jpeg;base64,{img_64}";
        var l = {{ t: {m_t}, b: {m_b}, sx: {m_l}, dx: {m_r} }};
        var active = null;
        img.onload = function() {{ c.width = img.width; c.height = img.height; draw(); }};
        c.onmousedown = function(e) {{
            var r = c.getBoundingClientRect(); var x = e.clientX - r.left, y = e.clientY - r.top;
            if (Math.abs(y - l.t) < 15) active = 't'; else if (Math.abs(y - l.b) < 15) active = 'b';
            else if (Math.abs(x - l.sx) < 15) active = 'sx'; else if (Math.abs(x - l.dx) < 15) active = 'dx';
        }};
        c.onmousemove = function(e) {{
            if (!active) return; var r = c.getBoundingClientRect();
            if (active == 't' || active == 'b') l[active] = e.clientY - r.top; else l[active] = e.clientX - r.left;
            draw();
        }};
        c.onmouseup = function() {{ active = null; }};
        function draw() {{
            ctx.drawImage(img, 0, 0); ctx.strokeStyle = '#00f2ff'; ctx.lineWidth = 3; ctx.setLineDash([10, 5]);
            ctx.beginPath();
            ctx.moveTo(0, l.t); ctx.lineTo(c.width, l.t); ctx.moveTo(0, l.b); ctx.lineTo(c.width, l.b);
            ctx.moveTo(l.sx, 0); ctx.lineTo(l.sx, c.height); ctx.moveTo(l.dx, 0); ctx.lineTo(l.dx, c.height);
            ctx.stroke();
        }}
        document.getElementById('run').onclick = function() {{
            google.colab.kernel.invokeFunction('notebook.AnalisiMorfologica', [l], {{}});
        }};
    </script>
    """
    with app_out:
        clear_output(wait=True); display(widgets.HTML(html))

def AnalisiMorfologica(muri):
    with app_out:
        clear_output(wait=True); print("Elaborazione filtri non lineari in corso...")

        y1, y2 = int(muri['t'] / ratio_disp), int(muri['b'] / ratio_disp)
        x1, x2 = int(muri['sx'] / ratio_disp), int(muri['dx'] / ratio_disp)

        y1, y2 = sorted([y1, y2]); x1, x2 = sorted([x1, x2])
        y1, y2 = max(0, y1), min(gray_ready.shape[0], y2)
        x1, x2 = max(0, x1), min(gray_ready.shape[1], x2)

        recinto = gray_ready[y1:y2, x1:x2]; hr, wr = recinto.shape

        # 1. Calcolo diga infallibile (chiusura morfologica + inviluppo convesso)
        kernel_diga = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (45, 45))
        diga_chiusa = cv2.morphologyEx(recinto, cv2.MORPH_CLOSE, kernel_diga)
        _, m_red_base = cv2.threshold(diga_chiusa, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        m_red = np.zeros_like(m_red_base)
        contours, _ = cv2.findContours(m_red_base, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            c_max = max(contours, key=cv2.contourArea)
            hull = cv2.convexHull(c_max)
            cv2.drawContours(m_red, [hull], -1, 255, -1)

        # 2. Calcolo ghiandole (top-hat + soglia adattiva per basso contrasto)
        clahe_g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        ghia_eq = clahe_g.apply(recinto)

        kernel_tubi = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 25))
        tophat = cv2.morphologyEx(ghia_eq, cv2.MORPH_TOPHAT, kernel_tubi)

        # La soglia adattiva analizza piccole aree (21x21). Se il contrasto è piatto, lo ignora.
        m_semi = cv2.adaptiveThreshold(tophat, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, -3)

        # Pulizia finale e intersezione con la diga
        m_semi_pulita = cv2.morphologyEx(m_semi, cv2.MORPH_OPEN, np.ones((3,1), np.uint8))
        m_ghia = cv2.bitwise_and(m_semi_pulita, m_red)

        def to_png_64(mask, color_bgr):
            canvas = np.zeros((hr, wr, 4), dtype=np.uint8)
            canvas[mask > 0, 0:3] = color_bgr; canvas[mask > 0, 3] = 255
            _, enc = cv2.imencode('.png', canvas); return base64.b64encode(enc).decode('utf-8')

        red_64, ghia_64 = to_png_64(m_red, [0, 0, 255]), to_png_64(m_ghia, [0, 255, 0])
        _, i_enc = cv2.imencode('.jpg', cv2.cvtColor(img_disp, cv2.COLOR_RGB2BGR))

        html_report = f"""
        <div style="background: #1a1a1a; padding: 20px; border-radius: 12px; color: white; display: inline-block; font-family: sans-serif;">
            <div style="position: relative; display: inline-block; cursor: none;">
                <canvas id="resC"></canvas>
            </div>
            <div style="margin-top: 15px; background: #333; padding: 20px; border-radius: 8px;">
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
                    <div>
                        Strumento selezionato:<br>
                        <select id="toolType" style="padding: 10px; width: 100%; background: #444; color: white; border: none; border-radius: 4px; margin-top:5px;">
                            <option value="sub_both">Cancella tutto (verde e rosso)</option>
                            <option value="sub_g">Cancella solo verde</option>
                            <option value="add_g">Disegna verde</option>
                            <option value="sub_r">Cancella solo rosso</option>
                            <option value="add_r">Disegna rosso</option>
                        </select>
                    </div>
                    <div>
                        Diametro punta:<br>
                        <input type="range" id="bSize" min="2" max="60" value="20" style="width: 100%; margin-top:10px;">
                    </div>
                </div>
                <div style="margin-top: 20px; display: flex; gap: 10px;">
                    <button id="reset" style="flex:1; padding: 12px; background: #f39c12; color: white; border: none; border-radius: 4px; cursor: pointer;">Ripristina analisi base</button>
                    <button id="back" style="flex:1; padding: 12px; background: #c0392b; color: white; border: none; border-radius: 4px; cursor: pointer;">Torna alla calibrazione muri</button>
                </div>
                <div id="score" style="margin-top: 25px; padding: 15px; background: #111; border-radius: 8px; font-size: 20px; color: #2ecc71; border-left: 5px solid #2ecc71;">Calcolo in corso...</div>
                <canvas id="scale" width="700" height="50" style="margin-top:10px;"></canvas>
            </div>
        </div>
        <script>
            var c = document.getElementById('resC'); var ctx = c.getContext('2d');
            var sc = document.getElementById('scale'); var sctx = sc.getContext('2d');
            var bImg = new Image(); bImg.src = "data:image/jpeg;base64,{base64.b64encode(i_enc).decode('utf-8')}";
            var rM = new Image(); rM.src = "data:image/png;base64,{red_64}";
            var gM = new Image(); gM.src = "data:image/png;base64,{ghia_64}";
            var rC = document.createElement('canvas'); var gC = document.createElement('canvas');
            var rX = rC.getContext('2d'); var gX = gC.getContext('2d');
            var mx=0, my=0, l=0;
            function check() {{ if(++l == 3) reset(); }}
            bImg.onload=check; rM.onload=check; gM.onload=check;
            function reset() {{
                c.width = bImg.width; c.height = bImg.height;
                rC.width = gC.width = {wr}; rC.height = gC.height = {hr};
                rX.drawImage(rM,0,0); gX.drawImage(gM,0,0);
                up();
            }}
            c.onmousemove = function(e) {{
                var rect = c.getBoundingClientRect(); mx = e.clientX - rect.left; my = e.clientY - rect.top;
                if (e.buttons === 1) {{
                    var rx = (mx - {muri['sx']}) * ({wr}/{muri['dx']-muri['sx']});
                    var ry = (my - {muri['t']}) * ({hr}/{muri['b']-muri['t']});
                    var sz = document.getElementById('bSize').value;
                    var tool = document.getElementById('toolType').value;
                    if (tool == 'sub_both') {{
                        gX.globalCompositeOperation = 'destination-out'; gX.beginPath(); gX.arc(rx, ry, sz, 0, Math.PI*2); gX.fill();
                        rX.globalCompositeOperation = 'destination-out'; rX.beginPath(); rX.arc(rx, ry, sz, 0, Math.PI*2); rX.fill();
                    }} else if (tool == 'sub_g') {{
                        gX.globalCompositeOperation = 'destination-out'; gX.beginPath(); gX.arc(rx, ry, sz, 0, Math.PI*2); gX.fill();
                    }} else if (tool == 'add_g') {{
                        gX.globalCompositeOperation = 'source-over'; gX.fillStyle = 'rgba(0,255,0,1)'; gX.beginPath(); gX.arc(rx, ry, sz, 0, Math.PI*2); gX.fill();
                    }} else if (tool == 'sub_r') {{
                        rX.globalCompositeOperation = 'destination-out'; rX.beginPath(); rX.arc(rx, ry, sz, 0, Math.PI*2); rX.fill();
                    }} else if (tool == 'add_r') {{
                        rX.globalCompositeOperation = 'source-over'; rX.fillStyle = 'rgba(255,0,0,1)'; rX.beginPath(); rX.arc(rx, ry, sz, 0, Math.PI*2); rX.fill();
                    }}
                }}
                up();
            }};
            function up() {{
                ctx.clearRect(0,0,c.width,c.height); ctx.drawImage(bImg,0,0);
                function drawL(canv, col, alp) {{
                    var t = document.createElement('canvas'); t.width = c.width; t.height = c.height;
                    var tx = t.getContext('2d');
                    tx.drawImage(canv, {muri['sx']}, {muri['t']}, {muri['dx']-muri['sx']}, {muri['b']-muri['t']});
                    tx.globalCompositeOperation = 'source-in'; tx.fillStyle = col; tx.fillRect(0,0,t.width,t.height);
                    ctx.save(); ctx.globalAlpha = alp; ctx.drawImage(t, 0, 0); ctx.restore();
                }}
                drawL(rC, "red", 0.18); drawL(gC, "#00ff00", 0.30);
                var tool = document.getElementById('toolType').value;
                ctx.strokeStyle = tool.includes('both') ? "white" : (tool.includes('g') ? "#00ff00" : "red");
                ctx.lineWidth = 2; ctx.beginPath(); ctx.arc(mx, my, document.getElementById('bSize').value * (({muri['dx']-muri['sx']})/{wr}), 0, Math.PI*2); ctx.stroke();
                calc();
            }}
            function calc() {{
                var rD = rX.getImageData(0,0,{wr},{hr}).data;
                var gD = gX.getImageData(0,0,{wr},{hr}).data;
                var rc=0, gc=0; for(var i=3; i<rD.length; i+=4) {{ if(rD[i]>10) rc++; }}
                for(var i=3; i<gD.length; i+=4) {{ if(gD[i]>10 && rD[i]>10) gc++; }}
                var p = (1 - (gc/rc)) * 100; if(rc==0) p=0;
                document.getElementById('score').innerHTML = "Risultato: grado " + (p<=25?0:p<=50?1:p<=75?2:3) + " - " + p.toFixed(1) + "% di perdita ghiandolare";
                sctx.clearRect(0,0,700,50);
                var grd = sctx.createLinearGradient(0,0,700,0);
                grd.addColorStop(0,"#2ecc71"); grd.addColorStop(0.5,"#f1c40f"); grd.addColorStop(1,"#e74c3c");
                sctx.fillStyle=grd; sctx.fillRect(0,15,700,20);
                var x=(p/100)*700; sctx.fillStyle="white"; sctx.fillRect(x-2,5,4,40);
            }}
            document.getElementById('reset').onclick = reset;
            document.getElementById('back').onclick = function() {{ google.colab.kernel.invokeFunction('notebook.Muri', [], {{}}); }};
        </script>
        """
        display(widgets.HTML(html_report))

output.register_callback('notebook.Muri', mostra_schermata_muri)
output.register_callback('notebook.AnalisiMorfologica', AnalisiMorfologica)
display(app_out); mostra_schermata_muri()

In [ ]:
# @title v3.4 - stabilizzazione ottica
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive, output
import ipywidgets as widgets
import base64
from IPython.display import display, clear_output

# Setup e caricamento dell'immagine
if not os.path.exists('/content/drive'): drive.mount('/content/drive')
work_dir = '/content/drive/MyDrive/MeibomioGrafia/immagini_IR'
file_list = sorted([f for f in os.listdir(work_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

for i, f in enumerate(file_list): print(f"[{i}] {f}")
scelta = int(input("\n👉 Simone, inserisci il numero della foto da analizzare: "))
path_img = os.path.join(work_dir, file_list[scelta])

img_raw = cv2.imread(path_img)
h_orig, w_orig = img_raw.shape[:2]
w_an = 1024
ratio_an = w_an / w_orig
img_an = cv2.resize(img_raw, (w_an, int(h_orig * ratio_an)))
gray_an = cv2.cvtColor(img_an, cv2.COLOR_BGR2GRAY)

# Calcolo muri sull'immagine grezza pre-trattata
_, gray_ready = cv2.threshold(gray_an, 235, 235, cv2.THRESH_TRUNC)
gray_ready = cv2.medianBlur(gray_ready, 5)

w_disp = 750
ratio_disp = w_disp / w_an
img_disp_rgb = cv2.cvtColor(gray_ready, cv2.COLOR_GRAY2RGB)
img_disp = cv2.resize(img_disp_rgb, (w_disp, int(img_disp_rgb.shape[0] * ratio_disp)))

try:
    blur_m = cv2.GaussianBlur(gray_ready, (21, 21), 0)
    lx = np.mean(blur_m, axis=0); xm = np.argmax(lx); sx_v = np.max(lx) * 0.70
    l_a, r_a = 0, w_an-1
    for x in range(xm, 0, -1):
        if lx[x] < sx_v: l_a = x; break
    for x in range(xm, w_an):
        if lx[x] < sx_v: r_a = x; break

    fy = np.mean(blur_m[:, l_a:r_a], axis=1); py = np.gradient(fy.astype(float))
    yp = np.argmax(fy); msop, msot = np.argmax(py[:yp]), np.argmin(py[yp:]) + yp
    alt = msot - msop
    m_t_an, m_b_an = int(msop + int(alt*0.07)), int(msot - int(alt*0.05))
except Exception:
    print("Avviso: applico muri standard di sicurezza.")
    m_t_an, m_b_an = int(gray_ready.shape[0]*0.2), int(gray_ready.shape[0]*0.8)
    l_a, r_a = int(w_an*0.2), int(w_an*0.8)

m_t, m_b = int(m_t_an * ratio_disp), int(m_b_an * ratio_disp)
m_l, m_r = int(l_a * ratio_disp), int(r_a * ratio_disp)

app_out = widgets.Output()

# Interfaccia muri
def mostra_schermata_muri():
    _, buffer = cv2.imencode('.jpg', cv2.cvtColor(img_disp, cv2.COLOR_RGB2BGR))
    img_64 = base64.b64encode(buffer).decode('utf-8')
    html = f"""
    <div style="border: 4px solid #444; padding: 10px; display: inline-block; background: #1a1a1a; border-radius: 10px;">
        <canvas id="meibo"></canvas>
        <div style="margin-top: 10px;">
            <button id="run" style="padding: 15px 30px; background: #007bff; color: white; border: none; cursor: pointer; border-radius: 5px; width: 100%; font-weight: bold;">
                Avvia analisi strutturale avanzata
            </button>
        </div>
    </div>
    <script>
        var c = document.getElementById('meibo'); var ctx = c.getContext('2d');
        var img = new Image(); img.src = "data:image/jpeg;base64,{img_64}";
        var l = {{ t: {m_t}, b: {m_b}, sx: {m_l}, dx: {m_r} }};
        var active = null;
        img.onload = function() {{ c.width = img.width; c.height = img.height; draw(); }};
        c.onmousedown = function(e) {{
            var r = c.getBoundingClientRect(); var x = e.clientX - r.left, y = e.clientY - r.top;
            if (Math.abs(y - l.t) < 15) active = 't'; else if (Math.abs(y - l.b) < 15) active = 'b';
            else if (Math.abs(x - l.sx) < 15) active = 'sx'; else if (Math.abs(x - l.dx) < 15) active = 'dx';
        }};
        c.onmousemove = function(e) {{
            if (!active) return; var r = c.getBoundingClientRect();
            if (active == 't' || active == 'b') l[active] = e.clientY - r.top; else l[active] = e.clientX - r.left;
            draw();
        }};
        c.onmouseup = function() {{ active = null; }};
        function draw() {{
            ctx.drawImage(img, 0, 0); ctx.strokeStyle = '#00f2ff'; ctx.lineWidth = 3; ctx.setLineDash([10, 5]);
            ctx.beginPath();
            ctx.moveTo(0, l.t); ctx.lineTo(c.width, l.t); ctx.moveTo(0, l.b); ctx.lineTo(c.width, l.b);
            ctx.moveTo(l.sx, 0); ctx.lineTo(l.sx, c.height); ctx.moveTo(l.dx, 0); ctx.lineTo(l.dx, c.height);
            ctx.stroke();
        }}
        document.getElementById('run').onclick = function() {{
            google.colab.kernel.invokeFunction('notebook.AnalisiMorfologica', [l], {{}});
        }};
    </script>
    """
    with app_out:
        clear_output(wait=True); display(widgets.HTML(html))

# Analisi morfologica e interfaccia strumenti
def AnalisiMorfologica(muri):
    with app_out:
        clear_output(wait=True); print("Calcolo bilanciamento ottico e filtri in corso...")

        y1, y2 = int(muri['t'] / ratio_disp), int(muri['b'] / ratio_disp)
        x1, x2 = int(muri['sx'] / ratio_disp), int(muri['dx'] / ratio_disp)

        y1, y2 = sorted([y1, y2]); x1, x2 = sorted([x1, x2])
        y1, y2 = max(0, y1), min(gray_ready.shape[0], y2)
        x1, x2 = max(0, x1), min(gray_ready.shape[1], x2)

        recinto_float = gray_ready[y1:y2, x1:x2].astype(np.float32)
        hr, wr = recinto_float.shape

        # Calibrazione ottica del fondo
        k_size = (wr // 2) | 1
        if k_size < 3: k_size = 3
        mappa_luce = cv2.GaussianBlur(recinto_float, (k_size, k_size), 0)
        img_piatta = recinto_float - mappa_luce + np.mean(recinto_float)

        # Parametri ottici standardizzati
        mu_target = 180.0
        sigma_target = 40.0
        mu_old = np.mean(img_piatta)
        sigma_old = np.std(img_piatta)
        if sigma_old == 0: sigma_old = 1.0

        recinto_eq = (img_piatta - mu_old) * (sigma_target / sigma_old) + mu_target
        recinto_eq = np.clip(recinto_eq, 0, 255).astype(np.uint8)

        # Diga morfologica
        kernel_diga = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (70, 70))
        diga_chiusa = cv2.morphologyEx(recinto_eq, cv2.MORPH_CLOSE, kernel_diga)
        _, m_red_base = cv2.threshold(diga_chiusa, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        m_red = np.zeros_like(m_red_base)
        contours, _ = cv2.findContours(m_red_base, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            c_max = max(contours, key=cv2.contourArea)
            hull = cv2.convexHull(c_max)
            cv2.drawContours(m_red, [hull], -1, 255, -1)

        # Estrazione frequenze (Fourier)
        f = np.fft.fftshift(np.fft.fft2(recinto_eq.astype(float)/255.0))
        mask_f = np.zeros((hr, wr), np.uint8)
        v_offset = int(hr * 0.07)
        mask_f[hr//2 - v_offset : hr//2 + v_offset, :] = 1

        ghia_f = np.abs(np.fft.ifft2(np.fft.ifftshift(f * mask_f)))
        ghia_n = cv2.normalize(ghia_f, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        _, m_semi = cv2.threshold(cv2.createCLAHE(clipLimit=3.0).apply(ghia_n), 115, 255, cv2.THRESH_BINARY)

        kernel_unione = np.ones((7,3), np.uint8)
        m_semi_pulita = cv2.morphologyEx(m_semi, cv2.MORPH_CLOSE, kernel_unione)
        m_ghia = cv2.bitwise_and(m_semi_pulita, m_red)

        # Rendering Javascript
        def to_png_64(mask, color_bgr):
            canvas = np.zeros((hr, wr, 4), dtype=np.uint8)
            canvas[mask > 0, 0:3] = color_bgr; canvas[mask > 0, 3] = 255
            _, enc = cv2.imencode('.png', canvas); return base64.b64encode(enc).decode('utf-8')

        red_64, ghia_64 = to_png_64(m_red, [0, 0, 255]), to_png_64(m_ghia, [0, 255, 0])
        _, i_enc = cv2.imencode('.jpg', cv2.cvtColor(img_disp, cv2.COLOR_RGB2BGR))

        html_report = f"""
        <div style="background: #1a1a1a; padding: 20px; border-radius: 12px; color: white; display: inline-block; font-family: sans-serif;">
            <div style="position: relative; display: inline-block; cursor: none;">
                <canvas id="resC"></canvas>
            </div>
            <div style="margin-top: 15px; background: #333; padding: 20px; border-radius: 8px;">
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
                    <div>
                        Strumento selezionato:<br>
                        <select id="toolType" style="padding: 10px; width: 100%; background: #444; color: white; border: none; border-radius: 4px; margin-top:5px;">
                            <option value="sub_both">Cancella tutto (verde e rosso)</option>
                            <option value="sub_g">Cancella solo verde</option>
                            <option value="add_g">Disegna verde</option>
                            <option value="sub_r">Cancella solo rosso</option>
                            <option value="add_r">Disegna rosso</option>
                        </select>
                    </div>
                    <div>
                        Diametro punta:<br>
                        <input type="range" id="bSize" min="2" max="60" value="20" style="width: 100%; margin-top:10px;">
                    </div>
                </div>
                <div style="margin-top: 20px; display: flex; gap: 10px;">
                    <button id="undo" style="flex:1; padding: 12px; background: #3498db; color: white; border: none; border-radius: 4px; cursor: pointer; font-weight: bold;">Annulla ultima azione</button>
                    <button id="reset" style="flex:1; padding: 12px; background: #f39c12; color: white; border: none; border-radius: 4px; cursor: pointer; font-weight: bold;">Ripristina analisi base</button>
                    <button id="back" style="flex:1; padding: 12px; background: #c0392b; color: white; border: none; border-radius: 4px; cursor: pointer; font-weight: bold;">Torna alla calibrazione muri</button>
                </div>
                <div id="score" style="margin-top: 25px; padding: 15px; background: #111; border-radius: 8px; font-size: 20px; color: #2ecc71; border-left: 5px solid #2ecc71; font-weight: bold;">Calcolo in corso...</div>
                <canvas id="scale" width="700" height="50" style="margin-top:10px;"></canvas>
            </div>
        </div>
        <script>
            var c = document.getElementById('resC'); var ctx = c.getContext('2d');
            var sc = document.getElementById('scale'); var sctx = sc.getContext('2d');
            var bImg = new Image(); bImg.src = "data:image/jpeg;base64,{base64.b64encode(i_enc).decode('utf-8')}";
            var rM = new Image(); rM.src = "data:image/png;base64,{red_64}";
            var gM = new Image(); gM.src = "data:image/png;base64,{ghia_64}";
            var rC = document.createElement('canvas'); var gC = document.createElement('canvas');
            var rX = rC.getContext('2d');
            var gX = gC.getContext('2d');
            var mx=0, my=0, l=0;
            var undoStack = [];
            var isDrawing = false;

            function check() {{ if(++l == 3) reset(); }}
            bImg.onload=check; rM.onload=check; gM.onload=check;

            function reset() {{
                undoStack = [];
                c.width = bImg.width; c.height = bImg.height;
                rC.width = gC.width = {wr}; rC.height = gC.height = {hr};
                rX.drawImage(rM,0,0); gX.drawImage(gM,0,0);
                up();
            }}

            function saveState() {{
                if(undoStack.length > 20) undoStack.shift();
                undoStack.push({{
                    r: rX.getImageData(0, 0, {wr}, {hr}),
                    g: gX.getImageData(0, 0, {wr}, {hr})
                }});
            }}

            c.onmousedown = function(e) {{
                if (e.button === 0) {{
                    var rect = c.getBoundingClientRect(); mx = e.clientX - rect.left; my = e.clientY - rect.top;
                    saveState();
                    isDrawing = true;
                    doTool();
                }}
            }};

            c.onmousemove = function(e) {{
                var rect = c.getBoundingClientRect(); mx = e.clientX - rect.left; my = e.clientY - rect.top;
                if (isDrawing) {{
                    doTool();
                }} else {{
                    up();
                }}
            }};

            c.onmouseup = function(e) {{ isDrawing = false; }};
            c.onmouseleave = function(e) {{ isDrawing = false; }};

            function doTool() {{
                var rx = (mx - {muri['sx']}) * ({wr}/{muri['dx']-muri['sx']});
                var ry = (my - {muri['t']}) * ({hr}/{muri['b']-muri['t']});
                var sz = document.getElementById('bSize').value;
                var tool = document.getElementById('toolType').value;

                if (tool == 'sub_both') {{
                    gX.globalCompositeOperation = 'destination-out'; gX.beginPath(); gX.arc(rx, ry, sz, 0, Math.PI*2); gX.fill();
                    rX.globalCompositeOperation = 'destination-out'; rX.beginPath(); rX.arc(rx, ry, sz, 0, Math.PI*2); rX.fill();
                }} else if (tool == 'sub_g') {{
                    gX.globalCompositeOperation = 'destination-out'; gX.beginPath(); gX.arc(rx, ry, sz, 0, Math.PI*2); gX.fill();
                }} else if (tool == 'add_g') {{
                    gX.globalCompositeOperation = 'source-over'; gX.fillStyle = 'rgba(0,255,0,1)'; gX.beginPath(); gX.arc(rx, ry, sz, 0, Math.PI*2); gX.fill();
                }} else if (tool == 'sub_r') {{
                    rX.globalCompositeOperation = 'destination-out'; rX.beginPath(); rX.arc(rx, ry, sz, 0, Math.PI*2); rX.fill();
                }} else if (tool == 'add_r') {{
                    rX.globalCompositeOperation = 'source-over'; rX.fillStyle = 'rgba(255,0,0,1)'; rX.beginPath(); rX.arc(rx, ry, sz, 0, Math.PI*2); rX.fill();
                }}
                up();
            }}

            function up() {{
                ctx.clearRect(0,0,c.width,c.height); ctx.drawImage(bImg,0,0);
                function drawL(canv, col, alp) {{
                    var t = document.createElement('canvas'); t.width = c.width; t.height = c.height;
                    var tx = t.getContext('2d');
                    tx.drawImage(canv, {muri['sx']}, {muri['t']}, {muri['dx']-muri['sx']}, {muri['b']-muri['t']});
                    tx.globalCompositeOperation = 'source-in'; tx.fillStyle = col; tx.fillRect(0,0,t.width,t.height);
                    ctx.save(); ctx.globalAlpha = alp; ctx.drawImage(t, 0, 0); ctx.restore();
                }}
                drawL(rC, "red", 0.18); drawL(gC, "#00ff00", 0.30);
                var tool = document.getElementById('toolType').value;
                ctx.strokeStyle = tool.includes('both') ? "white" : (tool.includes('g') ? "#00ff00" : "red");
                ctx.lineWidth = 2; ctx.beginPath(); ctx.arc(mx, my, document.getElementById('bSize').value * (({muri['dx']-muri['sx']})/{wr}), 0, Math.PI*2); ctx.stroke();
                calc();
            }}

            function calc() {{
                var rD = rX.getImageData(0,0,{wr},{hr}).data;
                var gD = gX.getImageData(0,0,{wr},{hr}).data;
                var rc=0, gc=0; for(var i=3; i<rD.length; i+=4) {{ if(rD[i]>10) rc++; }}
                for(var i=3; i<gD.length; i+=4) {{ if(gD[i]>10 && rD[i]>10) gc++; }}
                var p = (1 - (gc/rc)) * 100; if(rc==0) p=0;
                document.getElementById('score').innerHTML = "Risultato: grado " + (p<=25?0:p<=50?1:p<=75?2:3) + " - " + p.toFixed(1) + "% di perdita ghiandolare";
                sctx.clearRect(0,0,700,50);
                var grd = sctx.createLinearGradient(0,0,700,0);
                grd.addColorStop(0,"#2ecc71"); grd.addColorStop(0.5,"#f1c40f"); grd.addColorStop(1,"#e74c3c");
                sctx.fillStyle=grd; sctx.fillRect(0,15,700,20);
                var x=(p/100)*700; sctx.fillStyle="white"; sctx.fillRect(x-2,5,4,40);
            }}

            document.getElementById('undo').onclick = function() {{
                if(undoStack.length > 0) {{
                    var state = undoStack.pop();
                    rX.putImageData(state.r, 0, 0);
                    gX.putImageData(state.g, 0, 0);
                    up();
                }}
            }};

            document.getElementById('reset').onclick = reset;
            document.getElementById('back').onclick = function() {{ google.colab.kernel.invokeFunction('notebook.Muri', [], {{}}); }};
        </script>
        """
        display(widgets.HTML(html_report))

output.register_callback('notebook.Muri', mostra_schermata_muri)
output.register_callback('notebook.AnalisiMorfologica', AnalisiMorfologica)
display(app_out); mostra_schermata_muri()